# 📱 Notebook 2: Universal API vs. BFF — measured

In notebook 1 we saw that a BFF sends fewer bytes. Now let's measure **latency** (time until the client has its response) and show three more real-world BFF techniques:

1. **Sequential fan-out** (bad) vs. **parallel fan-out** (good).
2. **Hard failure** (bad) vs. **graceful degradation** (good) when a downstream service dies.
3. **No caching** (bad) vs. a **tiny BFF cache** (good) for expensive calls.

We'll fake the downstream services with `time.sleep` so you can see the effect without setting up Docker.

## 🛠️ Setup

```bash
cd 05-microservices/bff
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This lab uses **only the Python standard library** — no servers to start, no Docker. We simulate HTTP calls with plain Python functions so you can focus on the pattern.

## Step 1 — simulated downstream services with realistic latency

Each service sleeps a bit to represent a network round-trip + database query.

In [ ]:
import time, json, random
from concurrent.futures import ThreadPoolExecutor

random.seed(0)

def _slow(ms: int):
    time.sleep(ms / 1000)

def svc_user(uid):
    _slow(30)   # ~30 ms
    return {'id': uid, 'name': 'Ada', 'email': 'a@x.io', 'bio': 'x' * 200}

def svc_orders(uid):
    _slow(40)   # ~40 ms
    return [{'id': i, 'total': round(random.random() * 100, 2),
             'lines': ['line' + str(j) for j in range(5)]} for i in range(5)]

def svc_recs(uid, _flaky: bool = False):
    _slow(60)   # ~60 ms (the slowest)
    if _flaky and random.random() < 0.5:
        raise TimeoutError('recommendations service timed out')
    return [{'sku': f'sku{i}', 'score': round(random.random(), 2)} for i in range(20)]


## Step 2 — ⚠️ Bad: universal API, client calls everything sequentially

A phone running the "universal API" pattern has to call all three endpoints and wait for each one. Total latency ≈ sum of every call.

In [ ]:
def universal_client_flow(uid):
    """The client calls three endpoints one after another."""
    t0 = time.time()
    u = svc_user(uid)
    o = svc_orders(uid)
    r = svc_recs(uid)
    payload = {'user': u, 'orders': o, 'recs': r}
    elapsed = time.time() - t0
    return elapsed, len(json.dumps(payload))

t, b = universal_client_flow(1)
print(f'universal (sequential, from client): {t*1000:>5.0f} ms, {b:>6,} bytes shipped to mobile')


## Step 3 — ✅ Better: BFF fans out **in parallel**

A BFF runs in a data centre with a fat pipe to each service. It can call them concurrently and return a single response to the phone. Total latency ≈ the **slowest** call (not the sum).

In [ ]:
def mobile_bff_parallel(uid):
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=3) as ex:
        f_user = ex.submit(svc_user, uid)
        f_orders = ex.submit(svc_orders, uid)
        # Note: we don't even call svc_recs - mobile doesn't need it.
    user, orders = f_user.result(), f_orders.result()
    payload = {'name': user['name'], 'order_count': len(orders)}
    elapsed = time.time() - t0
    return elapsed, len(json.dumps(payload))

t, b = mobile_bff_parallel(1)
print(f'mobile BFF (parallel fan-out):       {t*1000:>5.0f} ms, {b:>6,} bytes shipped to mobile')


Two wins stack here:

- We **skipped** the slowest service (`svc_recs`, 60 ms) — the mobile screen doesn't need recommendations.
- Of the two remaining services, we ran them **in parallel**, so our latency is `max(30, 40) = 40 ms` instead of `30 + 40 = 70 ms`.

Bytes on the wire to the phone also drop dramatically because the BFF reshapes the response.

## Step 4 — ⚠️ Bad: one failure ruins the whole screen

Real services fail. If the recommendations service is down and the BFF treats it as fatal, **the entire page goes blank** — even though the user's name and orders loaded fine.

Below, `svc_recs` has a 50% chance of raising a `TimeoutError`. Run it a few times.

In [ ]:
def web_bff_fragile(uid):
    """Any failure crashes the whole response - BAD."""
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=3) as ex:
        f_user = ex.submit(svc_user, uid)
        f_orders = ex.submit(svc_orders, uid)
        f_recs = ex.submit(svc_recs, uid, True)   # flaky!
    payload = {
        'user': f_user.result(),
        'orders': f_orders.result(),
        'recs': f_recs.result(),   # if this throws, the whole request 500s
    }
    return time.time() - t0, payload

for attempt in range(3):
    try:
        t, _ = web_bff_fragile(1)
        print(f'attempt {attempt+1}: OK in {t*1000:.0f} ms')
    except Exception as e:
        print(f'attempt {attempt+1}: whole page FAILED - {e}')


## Step 5 — ✅ Better: graceful degradation

A good BFF treats **non-critical data as optional**. If recommendations fail, we still ship user + orders and let the UI render an empty "Recommended for you" shelf. The page is still useful.

In [ ]:
def web_bff_resilient(uid):
    """Essential data required; optional data falls back to [] on failure - GOOD."""
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=3) as ex:
        f_user = ex.submit(svc_user, uid)
        f_orders = ex.submit(svc_orders, uid)
        f_recs = ex.submit(svc_recs, uid, True)   # still flaky

        try:
            recs = f_recs.result(timeout=1.0)
        except Exception as e:
            recs = []   # graceful fallback
            degraded = str(e)
        else:
            degraded = None

    payload = {
        'user': f_user.result(),
        'orders': f_orders.result(),
        'recs': recs,
        '_degraded': degraded,   # tell the UI so it can hide the empty shelf
    }
    return time.time() - t0, payload

ok = degraded = 0
for _ in range(6):
    t, p = web_bff_resilient(1)
    if p['_degraded']:
        degraded += 1
    else:
        ok += 1
print(f'6 attempts: {ok} fully loaded, {degraded} served with empty recs (page still works)')


**The key idea:** a BFF is the natural place to implement *UI-specific* resilience rules. The downstream `svc_recs` team doesn't have to know that "empty list is OK for the web page" — the web BFF encodes that product decision.

## ⏱️ Side note — the **timeout budget**

Did you notice `f_recs.result(timeout=1.0)` above? That's a **timeout budget**: the BFF refuses to wait more than 1 second for a single downstream call, because the phone is already waiting for the whole response.

A healthy BFF gives each call a deliberate budget:

```
total page budget       = 500 ms   (how long the user will wait)
  └── svc_user budget   = 100 ms   (fast, required)
  └── svc_orders budget = 150 ms   (medium, required)
  └── svc_recs budget   = 200 ms   (slowest, optional — fall back on timeout)
  └── spare             =  50 ms   (network, JSON serialization, safety margin)
```

Without a budget, one slow service can hold up every other screen it touches (a classic *head-of-line blocking* bug).


## Step 6 — ✅ Even better: tiny in-process cache for hot data

Recommendations often don't change every second. A short TTL cache in the BFF turns most calls into near-instant hits, and shields the downstream service from traffic spikes.

In [ ]:
_cache: dict = {}
TTL_SECONDS = 2

def svc_recs_cached(uid):
    key = ('recs', uid)
    now = time.time()
    hit = _cache.get(key)
    if hit and now - hit['at'] < TTL_SECONDS:
        return hit['value']
    value = svc_recs(uid)            # slow 60 ms call
    _cache[key] = {'at': now, 'value': value}
    return value

t0 = time.time(); svc_recs_cached(1); first = time.time() - t0
t0 = time.time(); svc_recs_cached(1); second = time.time() - t0
print(f'first  call (miss): {first*1000:>6.1f} ms')
print(f'second call (hit):  {second*1000:>6.1f} ms  <- served from BFF memory')


## 📊 Side-by-side summary

| Approach                                  | Fan-out     | Failure handling | Caching   | Latency (mobile) |
|-------------------------------------------|-------------|------------------|-----------|------------------|
| ⚠️ Universal API, client-side calls       | Sequential  | All-or-nothing   | None      | ~130 ms          |
| ✅ BFF with parallel fan-out               | Parallel    | All-or-nothing   | None      | ~40 ms           |
| ✅✅ BFF + graceful degradation            | Parallel    | Partial is fine  | None      | ~40 ms (robust)  |
| ✅✅✅ BFF + parallel + fallback + cache | Parallel    | Partial is fine  | Short TTL | ~0 ms on hit     |

### 🧠 Takeaways

- A BFF's **physical location** (data centre, close to services) makes parallel fan-out cheap.
- A BFF is the right place to encode **client-specific resilience** rules.
- Small **request-level caches** in the BFF dramatically reduce downstream load and smooth spikes.
- Trade-offs: code duplication between BFFs, more services to operate, potential for BFFs to sprawl.

👉 Next: [`03_bff_vs_gateway_and_pitfalls.ipynb`](./03_bff_vs_gateway_and_pitfalls.ipynb) — when to use a BFF, when **not** to, and how to avoid the classic mistakes.